In [1]:
import tempfile
import pickle
import sys
import traceback
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from pprint import pprint

from one.api import ONE
from scipy.special import logit, softmax
import os
from os.path import join
import pickle as pkl
from brainwidemap.bwm_loading import bwm_query, bwm_units, load_trials_and_mask, merge_probes

import concurrent.futures
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
import sys

from one.api import ONE
from brainbox.io.one import SessionLoader
from iblatlas.atlas import BrainRegions
from sklearn.metrics import balanced_accuracy_score

from brainwidemap.bwm_loading import (
    bwm_query,
    load_good_units,
    load_trials_and_mask,
    merge_probes,
    bwm_units,
)

In [2]:
engagement_stats = pd.read_parquet("../data/external/merged_behavioral_and_states.pqt")

In [3]:
one = ONE()
bwm_df = bwm_units(one)

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01


In [4]:
# find significant untis

In [5]:
regions_of_interst = [
    "ACB",
    "IRN",
    "CA1",
    "SUV",
    "LSr",
    "DG",
    "PO",
    "CP",
    "APN",
    "SCm",
    "GRN",
    "SIM",
    "MG",
    "ACAd",
    "ZI",
    "VISa",
    "MRN",
    "VPL",
    "MD",
    "POL",
    "PARN",
    "SSp-bfd",
    "MV",
    "LP",
    "LGd",
    "RN",
    "SSs",
    "CA3",
    "PAG",
    "SI",
    "VPM",
    "PRNr",
    "PoT",
    "IP",
    "CENT3",
    "MOp",
    "LD",
    "RT",
    "PPN",
]

In [6]:
bwm_df_subset = bwm_df[bwm_df.Beryl.isin(regions_of_interst)]

In [7]:
df_region_list = bwm_df_subset.groupby(["Beryl"])["eid"].unique().reset_index()

In [20]:
eidx = engagement_stats['eid'][0]

In [24]:
engagement_subset = engagement_stats[engagement_stats['eid']==eidx]

In [25]:
engagement_subset

,animal,eid,contrastLeft,contrastRight,rewarded,probabilityLeft,p_state1,p_state2,signed_contrast
0,CSHL045,034e726f-b35f-41e0-8d6c-a22cc32391fb,1.0000,NaN,1.0,0.5,0.000021,0.999979,-1.0000
1,CSHL045,034e726f-b35f-41e0-8d6c-a22cc32391fb,NaN,0.125,-1.0,0.5,0.079605,0.920395,0.1250
2,CSHL045,034e726f-b35f-41e0-8d6c-a22cc32391fb,0.2500,NaN,1.0,0.5,0.365950,0.634050,-0.2500
3,CSHL045,034e726f-b35f-41e0-8d6c-a22cc32391fb,NaN,1.000,1.0,0.5,0.557431,0.442569,1.0000
4,CSHL045,034e726f-b35f-41e0-8d6c-a22cc32391fb,0.0000,NaN,1.0,0.5,0.600120,0.399880,0.0000
...,...,...,...,...,...,...,...,...,...
584,CSHL045,034e726f-b35f-41e0-8d6c-a22cc32391fb,0.1250,NaN,-1.0,0.8,0.601017,0.398983,-0.1250
585,CSHL045,034e726f-b35f-41e0-8d6c-a22cc32391fb,0.0625,NaN,-1.0,0.8,0.616611,0.383389,-0.0625
586,CSHL045,034e726f-b35f-41e0-8d6c-a22cc32391fb,1.0000,NaN,1.0,0.8,0.710953,0.289047,-1.0000
587,CSHL045,034e726f-b35f-41e0-8d6c-a22cc32391fb,1.0000,NaN,1.0,0.8,0.744542,0.255458,-1.0000


In [26]:
one_local = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    password="international",
    silent=True,
    username="intbrainlab",
    # mode="local",
)

In [27]:
trials, trial_mask = load_trials_and_mask(
            one_local, eidx, exclude_unbiased=True, exclude_nochoice=True
        )

(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/churchlandlab/Subjects/CSHL045/2020-02-24/001/alf/_ibl_trials.intervals_bpod.npy: 100%|██████████| 9.55k/9.55k [00:00<00:00, 31.8kB/s]
(S3) /Users/dkundu/Downloads/ONE/openalyx.internationalbrainlab.org/churchlandlab/Subjects/CSHL045/2020-02-24/001/alf/_ibl_trials.quiescencePeriod.npy: 100%|██████████| 4.84k/4.84k [00:00<00:00, 14.3kB/s]


In [34]:
result = trials.merge(engagement_stats[['p_state1','p_state2','signed_contrast']],left_index=True, right_index=True)

In [39]:
masked_result = result[trial_mask]

In [43]:
def get_trial_masks_engagement(trials, incon_only=False):
    """
    Returns boolean masks for 2 conditions (Incongruent, correct and disengaged).
    """
    masks = {}

    is_L_block = trials["probabilityLeft"] == 0.8
    is_R_block = trials["probabilityLeft"] == 0.2

    has_contrast_L = ~np.isnan(trials["contrastLeft"])
    has_contrast_R = ~np.isnan(trials["contrastRight"])
    is_engaged = trials['p_state1'] > trials['p_state2'] 



    # incongruent correct
    right_stimulus_incong_engaged = has_contrast_R & is_L_block & is_engaged
    left_stimulus_incong_engaged = has_contrast_L & is_R_block & is_engaged
    right_stimulus_incong_disengaged = has_contrast_R & is_L_block & ~is_engaged
    left_stimulus_incong_disengaged = has_contrast_L & is_R_block & ~is_engaged

    right_stimulus_cong_engaged = has_contrast_R & is_R_block & is_engaged
    left_stimulus_cong_engaged = has_contrast_L & is_L_block & is_engaged
    right_stimulus_cong_disengaged = has_contrast_R & is_R_block & ~is_engaged
    left_stimulus_cong_disengaged = has_contrast_L & is_L_block & ~is_engaged   


    incong_engaged = right_stimulus_incong_engaged | left_stimulus_incong_engaged
    incong_disengaged = right_stimulus_incong_disengaged | left_stimulus_incong_disengaged

    if not incon_only:
        cong_engaged = right_stimulus_cong_engaged | left_stimulus_cong_engaged
        cong_disengaged = right_stimulus_cong_disengaged | left_stimulus_cong_disengaged

    masks["Incongruent_engaged"] = incong_engaged
    masks["Incongruent_disengaged"] = incong_disengaged

    masks["Congruent_engaged"] = cong_engaged
    masks["Congruent_disengaged"] = cong_disengaged


    return masks, list(masks.keys())

In [44]:
masks, keys = get_trial_masks_engagement(masked_result)

In [47]:
[np.sum(masks[k]) for k in masks.keys()]

[np.int64(62), np.int64(9), np.int64(284), np.int64(21)]

In [48]:
from manifold.utils import get_trial_masks

In [49]:
congruency_masks, _ = get_trial_masks(masked_result)

In [50]:
[np.sum(congruency_masks[k]) for k in congruency_masks.keys()]

[np.int64(55), np.int64(16), np.int64(281), np.int64(24)]

False